In [1]:
# Generic imports
from collections import Counter
from warnings import filterwarnings
# filterwarnings(action='ignore')  # Uncoment if required

import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt


from sklearn.decomposition import NMF, PCA, LatentDirichletAllocation
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import rand_score

import spacy
from spacy import displacy

In [2]:
# Setting globally figure size in the notebook
plt.rcParams["figure.figsize"] = (8.0, 6.0)

# Brand colors dictionary
brand_colors = {
    "Denim": "#1358DB",
    "Supernova": "#FFCF00",
    # "Black": "#000000",
    # "Dark Blue": "#101D42",
    # "White": "#FFFFFF",
    "Dark Gray": "#6F6F6F",
    # "Light Gray 1": "#F6F6F4",
    "Light Gray 2": "#ADBDCC"
}

# Create a matplotlib colormap
brand_cmap = plt.cm.colors.ListedColormap(list(brand_colors.values()))
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=brand_colors.values())

# Numpy settings
np.random.seed(12345)

# Setting pandas display options
pd.set_option('display.max_colwidth', 200)

In [10]:
raw_df = pd.read_csv("C:/Users/klink/Downloads/year_2024.csv", low_memory=False)

In [11]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12229298 entries, 0 to 12229297
Data columns (total 99 columns):
 #   Column                                    Dtype  
---  ------                                    -----  
 0   activity_year                             int64  
 1   lei                                       object 
 2   derived_msa-md                            int64  
 3   state_code                                object 
 4   county_code                               float64
 5   census_tract                              object 
 6   conforming_loan_limit                     object 
 7   derived_loan_product_type                 object 
 8   derived_dwelling_category                 object 
 9   derived_ethnicity                         object 
 10  derived_race                              object 
 11  derived_sex                               object 
 12  action_taken                              int64  
 13  purchaser_type                            int64  
 14  

In [12]:
raw_df.head(10)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-2,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2024,549300DD5QQUHO6PCH70,38060,AZ,4013.0,04013061031,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,...,NaN,NaN,NaN,6566,36.72,101300,115,1398,1733,14
1,2024,549300DD5QQUHO6PCH70,24780,NC,37147.0,37147001100,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,5082,27.88,84700,106,1555,2198,32
2,2024,549300DD5QQUHO6PCH70,29414,IN,18127.0,18127050602,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4804,17.80,91100,122,1412,1736,39
3,2024,549300DD5QQUHO6PCH70,17900,SC,45063.0,45063020706,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,2977,79.27,86300,62,734,1184,41
4,2024,549300DD5QQUHO6PCH70,36420,OK,40081.0,40081961402,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4726,24.23,89100,94,1722,2135,26
5,2024,549300DD5QQUHO6PCH70,36740,FL,12095.0,12095012402,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,5741,84.46,90400,57,337,878,36
6,2024,549300DD5QQUHO6PCH70,99999,TX,48293.0,48293970600,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,5546,41.06,75500,93,1310,2129,40
7,2024,549300DD5QQUHO6PCH70,39300,RI,44009.0,44009050701,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,...,NaN,NaN,NaN,4166,8.04,113200,106,1235,1658,44
8,2024,549300DD5QQUHO6PCH70,99999,TX,48471.0,48471790302,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,7092,18.30,75500,123,1836,2487,32
9,2024,549300DD5QQUHO6PCH70,99999,MS,28081.0,28081950301,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4694,12.91,64400,155,1416,1797,27


In [33]:
rawcon_df = pd.read_csv("C:/Users/klink/Downloads/year_2024.csv", usecols=[0, 1, 2, 3, 4, 5, 7, 8,9,10,11,12,13,14,15,16,21,22,23,24,31,38,39,40,43,45,46,47,48,83,88,],low_memory=False)

In [34]:
rawcon_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12229298 entries, 0 to 12229297
Data columns (total 31 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   activity_year                   int64  
 1   lei                             object 
 2   derived_msa-md                  int64  
 3   state_code                      object 
 4   county_code                     float64
 5   census_tract                    object 
 6   derived_loan_product_type       object 
 7   derived_dwelling_category       object 
 8   derived_ethnicity               object 
 9   derived_race                    object 
 10  derived_sex                     object 
 11  action_taken                    int64  
 12  purchaser_type                  int64  
 13  preapproval                     int64  
 14  loan_type                       int64  
 15  loan_purpose                    int64  
 16  loan_amount                     float64
 17  loan_to_value_ratio      

In [35]:
pd.set_option('display.max_columns', None)
rawcon_df.head(10)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,derived_race,derived_sex,action_taken,purchaser_type,preapproval,loan_type,loan_purpose,loan_amount,loan_to_value_ratio,interest_rate,rate_spread,loan_term,property_value,construction_method,occupancy_type,total_units,income,debt_to_income_ratio,applicant_credit_score_type,co-applicant_credit_score_type,aus-1,denial_reason-1
0,2024,549300DD5QQUHO6PCH70,38060,AZ,4013.0,04013061031,VA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,Race Not Available,Sex Not Available,5,0,2,3,32,395000.0,NaN,NaN,NaN,360,NaN,1,1,1,65.0,NaN,9,9,2,10
1,2024,549300DD5QQUHO6PCH70,24780,NC,37147.0,37147001100,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,White,Female,1,2,2,3,1,305000.0,NaN,6.25,-0.304,360,345000,1,1,1,101.0,50%-60%,2,3,1,10
2,2024,549300DD5QQUHO6PCH70,29414,IN,18127.0,18127050602,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,White,Male,1,2,2,3,1,455000.0,NaN,6.625,-0.525,360,455000,1,1,2,105.0,50%-60%,1,10,1,10
3,2024,549300DD5QQUHO6PCH70,17900,SC,45063.0,45063020706,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,Black or African American,Joint,4,0,2,1,1,135000.0,NaN,NaN,NaN,360,NaN,1,3,1,216.0,NaN,9,9,1,10
4,2024,549300DD5QQUHO6PCH70,36420,OK,40081.0,40081961402,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,White,Joint,4,0,2,3,1,375000.0,NaN,NaN,NaN,360,NaN,1,1,1,147.0,NaN,9,9,1,10
5,2024,549300DD5QQUHO6PCH70,36740,FL,12095.0,12095012402,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,Black or African American,Female,1,71,2,3,1,165000.0,NaN,7.125,0.782,360,165000,1,1,1,65.0,50%-60%,3,10,1,10
6,2024,549300DD5QQUHO6PCH70,99999,TX,48293.0,48293970600,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,White,Male,1,2,2,3,1,205000.0,NaN,6.75,0.051,360,205000,1,1,1,70.0,50%-60%,1,10,1,10
7,2024,549300DD5QQUHO6PCH70,39300,RI,44009.0,44009050701,VA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,White,Male,3,0,2,3,32,375000.0,NaN,NaN,NaN,360,NaN,1,1,1,193.0,30%-<36%,3,10,6,3
8,2024,549300DD5QQUHO6PCH70,99999,TX,48471.0,48471790302,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,White,Joint,4,0,2,3,1,105000.0,NaN,NaN,NaN,360,NaN,1,1,1,73.0,NaN,9,9,1,10
9,2024,549300DD5QQUHO6PCH70,99999,MS,28081.0,28081950301,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,White,Joint,4,0,2,3,1,235000.0,NaN,NaN,NaN,360,NaN,1,1,1,93.0,NaN,9,9,2,10


In [36]:
print(rawcon_df['action_taken'].value_counts())

action_taken
1    6176052
3    2099966
4    1536699
6    1273313
5     581629
2     360560
8     153652
7      47427
Name: count, dtype: int64


In [38]:
# drop loans with Application withdrawn or closed for incompletness
rawcon_df = rawcon_df[~rawcon_df['action_taken'].isin([4, 5])]

In [39]:
print(rawcon_df['action_taken'].value_counts())

action_taken
1    6176052
3    2099966
6    1273313
2     360560
8     153652
7      47427
Name: count, dtype: int64


In [40]:
print(rawcon_df['occupancy_type'].value_counts())

occupancy_type
1    9165849
3     754832
2     190289
Name: count, dtype: int64


In [41]:
# drop loans that are not a primary residence
rawcon_df = rawcon_df[~rawcon_df['occupancy_type'].isin([2,3])]

In [42]:
print(rawcon_df['occupancy_type'].value_counts())

occupancy_type
1    9165849
Name: count, dtype: int64


In [43]:
print(rawcon_df['construction_method'].value_counts())

construction_method
1    8652694
2     513155
Name: count, dtype: int64


In [44]:
# drop manufactured homes 
rawcon_df = rawcon_df[~rawcon_df['construction_method'].isin([2])]

In [45]:
print(rawcon_df['construction_method'].value_counts())

construction_method
1    8652694
Name: count, dtype: int64


In [47]:
print(rawcon_df['loan_purpose'].value_counts())

loan_purpose
1     4548113
32    1139779
4     1088343
2      955806
31     904883
5       15770
Name: count, dtype: int64


In [48]:
# only include loans for a home purchase
rawcon_df = rawcon_df[~rawcon_df['loan_purpose'].isin([2, 31,32,4,5])]

In [49]:
print(rawcon_df['loan_purpose'].value_counts())

loan_purpose
1    4548113
Name: count, dtype: int64


In [53]:
print(rawcon_df['total_units'].value_counts())

total_units
1          4465632
2            64779
3            11309
4             6309
5-24            65
25-49            7
>149             6
50-99            5
100-149          1
Name: count, dtype: int64


In [54]:
# only include single unit homes 
rawcon_df = rawcon_df[rawcon_df['total_units'] == '1']

In [55]:
print(rawcon_df['total_units'].value_counts())

total_units
1    4465632
Name: count, dtype: int64


In [56]:
rawcon_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4465632 entries, 1 to 12229296
Data columns (total 31 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   activity_year                   int64  
 1   lei                             object 
 2   derived_msa-md                  int64  
 3   state_code                      object 
 4   county_code                     float64
 5   census_tract                    object 
 6   derived_loan_product_type       object 
 7   derived_dwelling_category       object 
 8   derived_ethnicity               object 
 9   derived_race                    object 
 10  derived_sex                     object 
 11  action_taken                    int64  
 12  purchaser_type                  int64  
 13  preapproval                     int64  
 14  loan_type                       int64  
 15  loan_purpose                    int64  
 16  loan_amount                     float64
 17  loan_to_value_ratio            

In [ ]:
# Data cleaned and removed non appliable information. Went from 12M to 4M records 

In [ ]:
rawcon_df.to_csv('C:/Users/klink/Downloads/HMDA_cleaned_data.csv', index=False)